In [1]:
# Importing modules
import pandas as pd
import numpy as np
from src.utils.smiles2morganfp import MorganFP
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Loading ESOL data
esol_data = pd.read_csv("data/train/ESOL.csv")

In [3]:
# Generate ESOL FP
esol_fp = MorganFP(esol_data["smiles"])
esol_fp["smiles"] = esol_fp.index
esol_fp = esol_fp.merge(esol_data, on="smiles")

In [4]:
# Models
model_dict = {"LR":LinearRegression(), 
		"ENet":ElasticNet(random_state=42), 
		"SVM":SVR(),
		"RF":RandomForestRegressor(random_state=42)}

# Params
params_dict = {

    'LR': {
        'fit_intercept': [True, False],
        'positive': [False, True]
    },

    'ENet': {
        'alpha': [0.001, 0.01, 0.1, 1.0],
        'l1_ratio': [0.2, 0.5, 0.8],
        'max_iter': [1000, 3000]
    },

    'SVM': {
        'kernel': ['rbf', 'linear'],
        'C': [0.1, 1, 10],
        'gamma': ['scale', 'auto'],
        'epsilon': [0.01, 0.1]
    },

    'RF': {
        'n_estimators': [100, 300],
        'max_depth': [None, 10, 30],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2],
        'max_features': ['sqrt', 'log2']
    }
}


# Function to run analysis
def RunML(data, dataName, modelName):
    
	# Feature Matrix (molecular fingerprint)
	X = data.drop(["smiles", "target"], axis=1)

	# Target labels (log solubility values)
	y = data["target"]

	# Train test split
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

	# Model
	model = GridSearchCV(model_dict[modelName], params_dict[modelName], cv=None, scoring="neg_root_mean_squared_error")

	# Model training
	model = model.fit(X_train.to_numpy(), y_train)

	# Model testing
	y_pred = model.predict(X_test.to_numpy())

	# Outfile to write results
	out = open(f"results/ML/output_{modelName}_{dataName}.txt", "a+")
	out.write(f'Model = {modelName} \nModel Params = {model.best_params_}\n')

	# Calculate MAE
	mae = mean_absolute_error(y_test, y_pred)
	out.write(f'MAE = {round(mae, 2)}\n')

	# Calculate RMSE
	rmse = root_mean_squared_error(y_test, y_pred)
	out.write(f'RMSE = {round(rmse, 2)}\n')

	r, p_val = pearsonr(y_test, y_pred)
	out.write(f'Pearson correlation coefficient = {round(r, 2)}, P-value= {round(p_val, 4)}\n')

	out.close()

In [5]:
# Model selection and hyperparameter tuning
for modelName in model_dict.keys():
    RunML(esol_fp, "ESOL", modelName)

In [6]:
# Parsing results
dfs = []
for model in model_dict.keys():
    data = open(f"results/ML/output_{model}_ESOL.txt").readlines()
    temp = []
    for line in data:
        if line[0]!= "P" and line.split(" ")[0] != "Model":
            temp.append(float(line.split("=")[1]))
        elif line[0]!= "P" and line.split(" ")[0] == "Model":
            temp.append(str(line.split("=")[1]).strip())
        else:
            temp.append(float(line.split("=")[1].split(",")[0]))
    df = pd.DataFrame(np.array(temp).reshape(1, 5), columns=["Model", "Model Params", "MAE", "RMSE", "r"])
    df["Model"] = model
    dfs.append(df)
final_df = pd.concat(dfs)
final_df.to_csv("results/Output_Hyperparameter_Optimization_ML.csv", quoting=False, index=False)

In [7]:
# Best parameters
for model in model_dict.keys():
    temp = final_df[final_df["Model"]==model]
    print(temp.sort_values(by=["RMSE"]).head(1),"\n")

  Model                                Model Params   MAE  RMSE     r
0    LR  {'fit_intercept': True, 'positive': False}  1.46  2.08  0.58 

  Model                                       Model Params   MAE  RMSE     r
0  ENet  {'alpha': 0.01, 'l1_ratio': 0.2, 'max_iter': 1...  0.95  1.23  0.77 

  Model                                       Model Params   MAE RMSE     r
0   SVM  {'C': 10, 'epsilon': 0.01, 'gamma': 'scale', '...  0.77  1.0  0.86 

  Model                                       Model Params   MAE  RMSE     r
0    RF  {'max_depth': None, 'max_features': 'sqrt', 'm...  0.86  1.14  0.81 

